In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Lipinski, Descriptors

if not os.path.exists('sascorer.py'):
    !wget https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/sascorer.py
if not os.path.exists('fpscores.pkl.gz'):
    !wget https://raw.githubusercontent.com/rdkit/rdkit/master/Contrib/SA_Score/fpscores.pkl.gz

import sascorer

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
ADMET_PATH  = Path("../data/link_invent_outputs/admet_test_protac_optimized_full.csv")
SCORED_PATH = Path("../data/link_invent_outputs/scored_sampling_test_protac_optimized_full.csv")
OUTPUT_ALL  = Path("../data/link_invent_outputs/protac_ranked_v2.csv")
OUTPUT_TOP = Path("../data/link_invent_outputs/protac_top_docking_v2.csv")

In [3]:
# ── 1. Load & Merge ─────────────────────────────────────────────────────────
admet  = pd.read_csv(ADMET_PATH)
scored = pd.read_csv(SCORED_PATH)

df = pd.merge(admet, scored[["SMILES", "Active_Class", "Active_Probability",
                              "Epistemic_Uncertainty"]],
              on="SMILES", how="inner")
print(f"Merged dataset: {len(df)} compounds")

Merged dataset: 703 compounds


In [4]:
# ── 2. Нормалізація alert-колонок (float → int) ─────────────────────────────
for col in ["PAINS_alert", "BRENK_alert", "AMES", "hERG"]:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

In [5]:
# ── 3. Лінкерні дескриптори (RDKit) ─────────────────────────────────────────
def analyze_linker(smi):
    if pd.isna(smi):
        return None, None, None
    mol = Chem.MolFromSmiles(smi)
    if not mol:
        return None, None, None
    return (mol.GetNumHeavyAtoms(),
            Lipinski.FractionCSP3(mol),
            Descriptors.NumRotatableBonds(mol))

df[["Linker_Length", "Linker_fCsp3", "Linker_RotBonds"]] = df["Linker"].apply(
    lambda x: pd.Series(analyze_linker(x))
)
print(f"Linker descriptors computed. NaN rows: {df['Linker_Length'].isna().sum()}")

Linker descriptors computed. NaN rows: 0


In [6]:
# ── 4. SA Score (RDKit sascorer) ─────────────────────────────────────────────
def calc_sa(smi):
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(smi)
    return sascorer.calculateScore(mol) if mol else None

df["SA_Score"] = df["SMILES"].apply(calc_sa)
print(f"SA Score computed. NaN rows: {df['SA_Score'].isna().sum()}")

SA Score computed. NaN rows: 0


In [13]:
# ── 5. Hard Filters ──────────────────────────────────────────────────────────
filters = {
    "hERG"        : df["hERG"]         == 0,
    "AMES"        : df["AMES"]         == 0,
    "PAINS"       : df["PAINS_alert"]  == 0,
    "LogP"        : df["logP"]         <= 6.0,
    "Linker_fCsp3": df["Linker_fCsp3"] >= 0.4,
    "Active"      : df["Active_Probability"] >= 0.75,
    "Epistemic"   : df["Epistemic_Uncertainty"] < 0.30,
}

# Діагностика
print(f"\n── Filter diagnostics (n={len(df)}) ──")
cumulative = pd.Series(True, index=df.index)
for name, cond in filters.items():
    cumulative &= cond
    print(f"  {name:20s}  pass={cond.sum():4d}  cumulative={cumulative.sum():4d}")

df_f = df[cumulative].copy()
print(f"\nAfter hard filters: {len(df_f)} compounds")


── Filter diagnostics (n=703) ──
  hERG                  pass= 703  cumulative= 703
  AMES                  pass= 703  cumulative= 703
  PAINS                 pass= 689  cumulative= 689
  LogP                  pass= 396  cumulative= 383
  Linker_fCsp3          pass= 697  cumulative= 379
  Active                pass= 317  cumulative= 153
  Epistemic             pass= 551  cumulative=  89

After hard filters: 89 compounds


In [14]:
# ── 6. Нормалізація ──────────────────────────────────────────────────────────
def minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else pd.Series(0.5, index=s.index)

def invert(s):
    return 1 - minmax(s)

df_f["norm_active_prob"] = minmax(df_f["Active_Probability"])
df_f["norm_caco2"]       = minmax(df_f["Caco2_Wang"])
df_f["norm_sa"]          = invert(df_f["SA_Score"])        # менше = синтетично доступніше
df_f["norm_logp"]        = np.exp(-0.5 * ((df_f["logP"] - 5.5) / 1.5) ** 2)  # Gaussian, оптимум 5.5
df_f["norm_uncertainty"] = invert(df_f["Epistemic_Uncertainty"])

In [15]:
# ── 7. Composite Score ───────────────────────────────────────────────────────
weights = {
    "norm_active_prob" : 0.60,
    "norm_caco2"       : 0.10,
    "norm_sa"          : 0.10, 
    "norm_logp"        : 0.10,
    "norm_uncertainty" : 0.10,
}
# Сума ваг = 1.0 ✓

df_f["composite_score"] = sum(w * df_f[col] for col, w in weights.items())

# Score breakdown (для інтерпретації)
for col, w in weights.items():
    df_f[f"contrib_{col}"] = w * df_f[col]

In [16]:
# ── 8. Ранжування ────────────────────────────────────────────────────────────
df_ranked = df_f.sort_values("composite_score", ascending=False).reset_index(drop=True)
df_ranked["rank"] = df_ranked.index + 1

# Вихідні колонки (оригінальні значення + scoring)
out_cols = [
    "rank", "SMILES", "Warheads", "Linker",
    "composite_score",
    # Scoring компоненти (оригінальні значення)
    "Active_Probability", "Epistemic_Uncertainty",
    "Caco2_Wang", "SA_Score", "logP", "Linker_fCsp3",
    "BRENK_alert",
    # Лінкер
    "Linker_Length", "Linker_RotBonds",
    # Фіз-хім
    "molecular_weight", "tpsa", "hydrogen_bond_donors", "hydrogen_bond_acceptors",
    # Safety (hard filters)
    "hERG", "AMES", "PAINS_alert",
    # Link-INVENT
    "NLL",
    # Score contributions
    *[f"contrib_{c}" for c in weights],
]
out_cols = [c for c in out_cols if c in df_ranked.columns]
df_ranked[out_cols].to_csv(OUTPUT_ALL, index=False)
print(f"Saved {len(df_ranked)} ranked compounds → {OUTPUT_ALL}")

Saved 89 ranked compounds → ../data/link_invent_outputs/protac_ranked_v2.csv


In [17]:
# ── 9. Top-5 Diverse via Linker Murcko Scaffold Clustering ──────────────────
from rdkit.Chem import DataStructs, Scaffolds
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

def get_linker_scaffold(smi):
    """Murcko scaffold лінкера (не повного PROTACа)."""
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(smi)
    if not mol:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold) if scaffold else Chem.MolToSmiles(mol)

print(f"\n── Best candidate per Murcko scaffold ──")

pool = df_ranked.head(200).copy()
pool["linker_scaffold"] = pool["Linker"].apply(get_linker_scaffold)

# Найкращий за composite_score з кожного scaffold
best_per_scaffold = (pool
                     .groupby("linker_scaffold", sort=False)
                     .first()
                     .reset_index()
                     .sort_values("composite_score", ascending=False)
                     .reset_index(drop=True))

best_per_scaffold["docking_priority"] = best_per_scaffold.index + 1
print(f"  Unique scaffolds: {len(best_per_scaffold)}")

out = [c for c in out_cols + ["linker_scaffold"] if c in best_per_scaffold.columns]
best_per_scaffold[out].to_csv(OUTPUT_TOP, index=False)
print(f"Saved {len(best_per_scaffold)} candidates → {OUTPUT_TOP}")

display_cols = ["docking_priority", "linker_scaffold", "composite_score",
                "Active_Probability", "Caco2_Wang", "SA_Score", "logP", "Linker_fCsp3"]
display_cols = [c for c in display_cols if c in best_per_scaffold.columns]
print(f"\n{'='*75}")
print("BEST CANDIDATE PER SCAFFOLD (sorted by composite score)")
print(f"{'='*75}")
print(best_per_scaffold[display_cols].to_string(index=False))


── Best candidate per Murcko scaffold ──
  Unique scaffolds: 26
Saved 26 candidates → ../data/link_invent_outputs/protac_top_docking_v2.csv

BEST CANDIDATE PER SCAFFOLD (sorted by composite score)
 docking_priority      linker_scaffold  composite_score  Active_Probability  Caco2_Wang  SA_Score    logP  Linker_fCsp3
                1 c1c[nH]c(C2CCCCC2)c1         0.792875            0.783585   -5.781943  4.054463 5.32090      0.636364
                2    c1ccn(C2CCCCC2)c1         0.730046            0.780932   -5.862730  4.036534 5.16520      0.636364
                3   c1ccc(C2CCCCC2)cc1         0.651810            0.775641   -5.607133  4.294138 5.98240      0.500000
                4   c1ccc(C2CCCNC2)cc1         0.638079            0.776779   -5.590581  4.428980 4.48890      0.571429
                5   c1ccn(CC2CCCCC2)c1         0.600362            0.769375   -5.508506  4.091869 4.89814      0.692308
                6 c1ccc(CCN2CCCCC2)cc1         0.596422            0.773972   -5.7